In [0]:
from pyspark.sql import functions as F

df_silver = spark.sql("SELECT * FROM files.amazon_files.amazon_silver")

In [0]:
df_com_categoria = df_silver.withColumn(
    "categoria_principal",
    F.split(F.col("category"), r"\|")[0]
)

In [0]:
# truncate = serve para não cortar dados longos escritos no código

df_com_categoria.select("category", "categoria_principal").show(10, truncate=False)

+---------------------------------------------------------------------------------+---------------------+
|category                                                                         |categoria_principal  |
+---------------------------------------------------------------------------------+---------------------+
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|
|Computers&Accessories|Accessories&Peripherals

In [0]:
#distinct = remove os valores repetidos

df_com_categoria.select("categoria_principal").distinct().count()

8

In [0]:
df_gold_categoria = df_com_categoria.groupBy("categoria_principal") \
    .agg(
        F.count("*").alias("qtd_produtos"),
        F.expr("percentile_approx(rating, 0.5)").alias("mediana_rating"),
        F.sum("rating_count").alias("total_reviews")
    ) \
        .orderBy(F.desc("mediana_rating"))

df_gold_categoria.show()

+--------------------+------------+--------------+-------------+
| categoria_principal|qtd_produtos|mediana_rating|total_reviews|
+--------------------+------------+--------------+-------------+
|      OfficeProducts|          30|           4.3|     142246.0|
|          Toys&Games|           1|           4.3|      15867.0|
|Computers&Accesso...|         447|           4.2|    7685363.0|
|         Electronics|         476|           4.1|  1.4902561E7|
|        Home&Kitchen|         394|           4.1|    2709966.0|
|     HomeImprovement|           2|           4.0|       8566.0|
|  MusicalInstruments|           2|           3.8|      88882.0|
|       Car&Motorbike|           1|           3.8|       1118.0|
+--------------------+------------+--------------+-------------+



In [0]:
# Uma função responsável por filtrar os dados por categoria principal
# - df_com_categoria = Spark DataFrame com todos os dados
# - categoria_principal = string com o nome da categoria principal
# - Retorna um Spark DataFrame com os dados filtrados por categoria principal

df_com_categoria.filter(F.col("categoria_principal") == "Electronics") \
    .select("product_name", "rating_count") \
    .orderBy(F.desc("rating_count")) \
    .show(10, truncate=False)

+------------------------------------------------------------------------------------------------------------+------------+
|product_name                                                                                                |rating_count|
+------------------------------------------------------------------------------------------------------------+------------+
|AmazonBasics Flexible Premium HDMI Cable (Black, 4K@60Hz, 18Gbps), 3-Foot                                   |426973.0    |
|Amazon Basics High-Speed HDMI Cable, 6 Feet (2-Pack),Black                                                  |426973.0    |
|Amazon Basics High-Speed HDMI Cable, 6 Feet - Supports Ethernet, 3D, 4K video,Black                         |426973.0    |
|AmazonBasics Flexible Premium HDMI Cable (Black, 4K@60Hz, 18Gbps), 3-Foot                                   |426972.0    |
|boAt Bassheads 100 in Ear Wired Earphones with Mic(Furious Red)                                             |363713.0    |
|boAt Ba

In [0]:
df_gold_categoria = df_com_categoria.groupBy("categoria_principal") \
    .agg(
        F.count("*").alias("qtd_produtos"),
        F.expr("percentile_approx(rating, 0.5)").alias("mediana_rating"),
        F.expr("percentile_approx(rating_count, 0.5)").alias("mediana_reviews")
    ) \
        .orderBy(F.desc("mediana_rating"))

df_gold_categoria.show()

+--------------------+------------+--------------+---------------+
| categoria_principal|qtd_produtos|mediana_rating|mediana_reviews|
+--------------------+------------+--------------+---------------+
|      OfficeProducts|          30|           4.3|         4308.0|
|          Toys&Games|           1|           4.3|        15867.0|
|Computers&Accesso...|         447|           4.2|         7732.0|
|         Electronics|         476|           4.1|         9378.0|
|        Home&Kitchen|         394|           4.1|         2288.0|
|     HomeImprovement|           2|           4.0|         2581.0|
|  MusicalInstruments|           2|           3.8|        20218.0|
|       Car&Motorbike|           1|           3.8|         1118.0|
+--------------------+------------+--------------+---------------+



In [0]:
df_gold_categoria.write.format("delta").mode("overwrite").saveAsTable("files.amazon_files.amazon_gold_categoria")

In [0]:
spark.sql("SELECT * FROM files.amazon_files.amazon_gold_categoria").show()

+--------------------+------------+--------------+---------------+
| categoria_principal|qtd_produtos|mediana_rating|mediana_reviews|
+--------------------+------------+--------------+---------------+
|      OfficeProducts|          30|           4.3|         4308.0|
|          Toys&Games|           1|           4.3|        15867.0|
|Computers&Accesso...|         447|           4.2|         7732.0|
|         Electronics|         476|           4.1|         9378.0|
|        Home&Kitchen|         394|           4.1|         2288.0|
|     HomeImprovement|           2|           4.0|         2581.0|
|       Car&Motorbike|           1|           3.8|         1118.0|
|  MusicalInstruments|           2|           3.8|        20218.0|
+--------------------+------------+--------------+---------------+

